# File

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv("ML_Ready_Extended_Geography.csv") # ETO YUNG GINAMIT KONG CSV YA CHINECK Q NA
X = df.drop('HIV_Status', axis=1)
y = df['HIV_Status']

# Splitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
df.head(10)

,Sex,Age_Group,Transmission,Healthcare_Access_Friction,Civil_Status,OFW_Status,Chemsex_Engagement,Alcohol_Sex_Risk,PrEP_Awareness,Transactional_Sex,STI_BBV_CoInfection_Any,HIV_Status
0,Female,<15,Male to Female Sex,2,Single,No,No,No,No,No TS,No,Reactive
1,Male,15-24,Male to Female Sex,2,Single,No,No,No,No,No TS,No,Reactive
2,Male,15-24,Male to Male Sex,2,Single,No,No,No,Yes,No TS,Yes,Reactive
3,Male,15-24,Male to Male Sex,2,Single,No,No,No,No,No TS,Yes,Reactive
4,Male,15-24,Male to Male Sex,2,Single,No,Yes,No,Yes,No TS,No,Reactive
5,Male,15-24,Male to Male/Female Sex,2,Single,No,No,Yes,Yes,Both,No,Reactive
6,Male,25-34,Male to Female Sex,2,Common-Law,No,No,No,No,No TS,No,Reactive
7,Male,25-34,Male to Male Sex,2,Single,No,No,No,Yes,No TS,No,Reactive
8,Male,25-34,Male to Male Sex,2,Single,No,No,No,No,No TS,No,Reactive
9,Male,25-34,Male to Male Sex,2,Single,No,No,Yes,Yes,Paid for sex,Yes,Reactive


# Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# list num and cat
numeric_features = []
categorical_features = ['Sex','Age_Group','Transmission',
                        'Healthcare_Access_Friction','Civil_Status','OFW_Status','Chemsex_Engagement',
                        'Alcohol_Sex_Risk','PrEP_Awareness','Transactional_Sex','STI_BBV_CoInfection_Any']

# 2. proprocessor
preprocessor = ColumnTransformer(
    transformers=[
        # Apply StandardScaler to numeric columns
        ('num', StandardScaler(), numeric_features),

        # Apply OneHotEncoder to categorical columns
        # handle_unknown='ignore' ensures the code won't crash if X_test has a category not seen in X_train
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # Keep any other columns not listed above (or use 'drop')
)

# 3. Fit and Transform X_train
# CRITICAL: We fit ONLY on X_train to learn the mean/std and categories.
# Then we transform both train and test using those learned values.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# convert into df
X# Convert into DataFrame AND preserve the original index
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index
)

# Dealing with y columns
mapping = {'Non-Reactive': 0, 'Reactive': 1}

# y_train and y_test already keep their indices, but it's good practice
# to ensure it's a Series or 1D array for Scikit-Learn classifiers
y_train_processed = y_train.map(mapping)
y_test_processed = y_test.map(mapping)

 # MODEL

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score, average_precision_score
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 1. Compute the balanced weights array
weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_processed),
    y=y_train_processed
)

# 2. Convert it into the dictionary format that Keras demands
class_weights = dict(enumerate(weights_array))
# 1. Define the Model Builder function using KerasTuner's "hp" (HyperParameters)
def build_model(hp):
    model = Sequential()

    # Let the tuner pick the number of neurons (32, 64, or 128)
    hp_neurons = hp.Int('neurons', min_value=32, max_value=128, step=32)
    # Let the tuner pick the dropout rate (0.2, 0.3, or 0.4)
    hp_dropout = hp.Float('dropout_rate', min_value=0.2, max_value=0.4, step=0.1)

    model.add(Dense(hp_neurons, activation='relu', input_shape=(X_train_processed.shape[1],)))
    model.add(BatchNormalization())
    model.add(Dropout(hp_dropout))

    model.add(Dense(hp_neurons // 2, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(hp_dropout))

    model.add(Dense(1, activation='sigmoid'))

    # Let the tuner pick the learning rate
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(curve='PR', name='auprc')] # Optimizing for minority class!
    )
    return model

# 2. Initialize the Grid Search Tuner
tuner = kt.GridSearch(
    build_model,
    objective=kt.Objective("val_auprc", direction="max"), # Maximize the PR-AUC
    max_trials=10, # Stop after trying 10 combinations to save time
    directory='nn_tuning_dir',
    project_name='reactive_prediction'
)

# Early stopping so bad models fail fast
early_stop = EarlyStopping(monitor='val_auprc', mode='max', patience=5)

# 3. Start the Search! 
print("Commencing KerasTuner Search...")
tuner.search(
    X_train_processed,
    y_train_processed,
    epochs=50,
    validation_split=0.2,
    class_weight=class_weights, # Pass your imbalanced weights!
    callbacks=[early_stop],
    verbose=1
)

# 4. Extract the Ultimate Winner
best_nn_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"\nBest Neurons: {best_nn_hps.get('neurons')}")
print(f"Best Dropout: {best_nn_hps.get('dropout_rate')}")
print(f"Best Learning Rate: {best_nn_hps.get('learning_rate')}")

# Build the final model with the best parameters
best_nn_model = tuner.hypermodel.build(best_nn_hps)

# Train the final model fully one last time
best_nn_model.fit(
    X_train_processed,
    y_train_processed,
    epochs=100,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[EarlyStopping(monitor='val_auprc', patience=15, restore_best_weights=True)],
    verbose=0
)

# 5. Evaluate
y_probs_nn = best_nn_model.predict(X_test_processed).ravel()
y_pred_nn = (y_probs_nn > 0.5).astype(int)

print("\n--- Tuned Neural Network Final Report ---")
print(classification_report(y_test_processed, y_pred_nn, target_names=['Non Reactive', 'Reactive']))
print(f"Tuned NN F1-Score: {f1_score(y_test_processed, y_pred_nn):.4f}")
print(f"Tuned NN AUPRC: {average_precision_score(y_test_processed, y_probs_nn):.4f}")

# Threshold Moving 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score
# 1. Get the raw probabilities from the NN
# Keras .predict() returns [[p1], [p2]...], so we flatten it to [p1, p2...]
nn_probs = best_nn_model.predict(X_test_processed).flatten()

# 2. Setup the search space
thresholds = np.linspace(0, 1, 100)
f1_scores = []
precisions = []
recalls = []

for t in thresholds:
    # Apply the threshold 't'
    preds = (nn_probs >= t).astype(int)

    f1_scores.append(f1_score(y_test_processed, preds))
    precisions.append(precision_score(y_test_processed, preds, zero_division=0))
    recalls.append(recall_score(y_test_processed, preds))

# 3. Find the "Golden Threshold"
best_idx = np.argmax(f1_scores)
miku_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

In [ ]:
# 4. Plotting the Threshold vs Metrics 
plt.figure(figsize=(10, 6))
plt.plot(thresholds, f1_scores, label='F1-Score', color='#39C5BB', linewidth=3) # Miku Teal
plt.plot(thresholds, precisions, label='Precision', color='#FFBACC', linestyle='--') # Luka Pink
plt.plot(thresholds, recalls, label='Recall', color='#FFCC11', linestyle='--')    # Rin Yellow

plt.axvline(miku_threshold, color='red', linestyle=':', label=f'Best Threshold: {miku_threshold:.2f}')

plt.title(f"Neural Network Threshold Optimization (Max F1: {best_f1:.4f})", fontsize=14)
plt.xlabel("Probability Threshold")
plt.ylabel("Score")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"--- NN Results ---")
print(f"Original F1 (50%): {f1_score(y_test_processed, (nn_probs >= 0.5).astype(int)):.4f}")
print(f"Optimized F1 ({miku_threshold*100:.0f}%): {best_f1:.4f}")

# 5. Calculate PR Curve data for the Neural Network
precision_nn, recall_nn, pr_thresholds_nn = precision_recall_curve(y_test_processed, nn_probs)
auprc_nn = average_precision_score(y_test_processed, nn_probs)

# 6. Plotting the Precision-Recall Curve (Continuing the Miku Theme!)
plt.figure(figsize=(10, 6))

# Plot the main curve and fill using Miku Teal
plt.plot(recall_nn, precision_nn, color='#39C5BB', linewidth=4, label=f'PR Curve (AUPRC = {auprc_nn:.4f})')
plt.fill_between(recall_nn, 0, precision_nn, color='#39C5BB', alpha=0.2)

# Mark the specific point on the curve where your "miku_threshold" sits
# We find the closest threshold in the PR curve array to your chosen optimal threshold
idx_nn = np.argmin(np.abs(pr_thresholds_nn - miku_threshold))
plt.scatter(recall_nn[idx_nn], precision_nn[idx_nn], color='red', s=100, zorder=5, label=f'Miku Threshold ({miku_threshold:.2f})')

plt.title("Neural Network Precision-Recall Curve\n(The 'Imbalance' Gold Standard)", fontsize=14)
plt.xlabel("Recall (Ability to find Reactives)")
plt.ylabel("Precision (Ability to be correct)")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# SHAP

In [ ]:
import shap
# 1. DeepExplainer needs a "Background Dataset" to compare against.
# We take a random sample of 100 rows from your training data to keep it fast!
# (Keras models usually prefer raw numpy arrays rather than DataFrames here)
background_data = X_train_processed.sample(n=100, random_state=42).values

# Initialize the Deep Explainer
explainer_nn = shap.DeepExplainer(best_nn_model, background_data)

# Calculate SHAP values for the test set
# We use .values to prevent TensorFlow shape errors
shap_vals_nn = explainer_nn.shap_values(X_test_processed.values)

# 2. Handle Keras Output Shapes
# DeepExplainer often returns a list or a 3D tensor depending on your output layer.
# We extract just the 2D array of values for the positive class and reshape it properly.
if isinstance(shap_vals_nn, list):
    shap_raw_nn = shap_vals_nn[0]
else:
    shap_raw_nn = shap_vals_nn

# Force it into a clean 2D shape to match your columns
shap_raw_nn = np.reshape(shap_raw_nn, X_test_processed.shape)

# Put them in a DataFrame for easy grouping
shap_df_nn = pd.DataFrame(shap_raw_nn, columns=X_test_processed.columns)
grouped_shap_df_nn = shap_df_nn.copy()

# 3. Define the exact prefixes of your One-Hot Encoded columns
categorical_prefixes = ['cat__Sex','cat__Age_Group','cat__Transmission',
                        'cat__Healthcare_Access_Friction','cat__Civil_Status','cat__OFW_Status','cat__Chemsex_Engagement',
                        'cat__Alcohol_Sex_Risk','cat__PrEP_Awareness','cat__Transactional_Sex','cat__STI_BBV_CoInfection_Any']

# 4. Stitch the dummy columns back together!
for prefix in categorical_prefixes:
    # Find all columns that start with this prefix
    dummy_cols = [col for col in X_test_processed.columns if col.startswith(f"{prefix}_")]

    if len(dummy_cols) > 0:
        # Sum the dummy parts to get the total category importance
        grouped_shap_df_nn[prefix] = shap_df_nn[dummy_cols].sum(axis=1)
        # Drop the diluted dummy columns
        grouped_shap_df_nn.drop(columns=dummy_cols, inplace=True)


In [ ]:
# 5. Plot the Global Feature Importance (Miku Teal!)
plt.figure(figsize=(12, 8))
plt.title(f"Neural Network: True Feature Importance (Grouped)", fontsize=14)

shap.summary_plot(
    grouped_shap_df_nn.values,
    features=grouped_shap_df_nn,
    feature_names=grouped_shap_df_nn.columns,
    plot_type="bar",
    color="#39C5BB",
    show=False
)

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## TOP 10 INDIVIDUAL FEATURES 

In [ ]:
explainer_nn = shap.DeepExplainer(best_nn_model, background_data)
shap_vals_nn = explainer_nn.shap_values(X_test_processed.values)

plt.figure(figsize=(10, 6))

raw_shap = shap_vals_nn[0] if isinstance(shap_vals_nn, list) else shap_vals_nn
if len(raw_shap.shape) == 3:
    raw_shap = raw_shap[:, :, 0]  # Try [:, :, 1] if your positive class is at index 1

shap.summary_plot(
    raw_shap,
    features=X_test_processed,
    feature_names=X_test_processed.columns,
    plot_type="bar",                        # Beeswarm style
    color="#39C5BB",
    max_display=10,
    show=False
)
plt.grid(axis='x', color='gray', alpha=0.2, linestyle=':')
plt.tight_layout()
plt.show()

## Beeswarm

In [ ]:
import matplotlib.colors as mcolors
plt.figure(figsize=(10, 6))
miku_cmap = mcolors.LinearSegmentedColormap.from_list("miku_gradient", ["#D1F2F0", "#39C5BB"])
shap.summary_plot(
    raw_shap,
    features=X_test_processed,
    feature_names=X_test_processed.columns,
    plot_type="dot",
    cmap=miku_cmap,
    max_display=10,
    show=False
)

plt.grid(axis='x', color='gray', alpha=0.2, linestyle=':')
plt.tight_layout()
plt.show()

# SAVING THE MODEL (RUN MO TO YA IF YOU THINK MAGANDA RESULT PARA MASAVE)

In [ ]:
model_filename_nn = 'best_nn_model.keras'
best_nn_model.save(model_filename_nn)

print(f"\nNeural Network successfully saved to {model_filename_nn}")